# ДЗ 5. Узагальнені табличні вирази (CTE) та проєктування ознак для ML

**Датасет:** Brazilian E-Commerce Public Dataset by Olist — [kaggle.com/datasets/olistbr/brazilian-ecommerce](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce).
Дані завантажуються через `kagglehub` (публічний датасет, ключ Kaggle не потрібен), тому stub-схема не використовується.

## 1. Setup і typed-схема Olist

In [1]:
import os, sys, subprocess, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

pkg = "pgserver" if sys.version_info < (3, 13) else "pixeltable-pgserver"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
                       pkg, "jupysql", "psycopg2-binary", "sqlalchemy", "pandas", "kagglehub"])

import pandas as pd
from sqlalchemy import create_engine, text

pgdata = Path("/tmp/hw5_pg")
if pkg == "pgserver":
    import pgserver
    pg = pgserver.get_server(pgdata, cleanup_mode="stop")
else:
    import pixeltable_pgserver
    pg = pixeltable_pgserver.get_server(pgdata, postgres_version=16)

engine = create_engine(pg.get_uri().replace("postgresql://", "postgresql+psycopg2://", 1), future=True)
with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())

%load_ext sql
%config SqlMagic.displaycon = False
%config SqlMagic.displaylimit = 0
%config SqlMagic.feedback = 0
%sql engine

PostgreSQL 16.14 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 14.2.1 20250110 (Red Hat 14.2.1-11), 64-bit


In [2]:
import kagglehub

dataset_dir = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
print(dataset_dir)

/home/user/.cache/kagglehub/datasets/olistbr/brazilian-ecommerce/versions/2


### Raw-таблиці

In [3]:
files = {
    "olist_customers_raw": "olist_customers_dataset.csv",
    "olist_orders_raw": "olist_orders_dataset.csv",
    "olist_order_items_raw": "olist_order_items_dataset.csv",
    "olist_products_raw": "olist_products_dataset.csv",
    "olist_sellers_raw": "olist_sellers_dataset.csv",
}

for table_name, file_name in files.items():
    df = pd.read_csv(os.path.join(dataset_dir, file_name))
    df.to_sql(table_name, engine, if_exists="replace", index=False, chunksize=10_000)
    print(table_name, df.shape)

olist_customers_raw (99441, 5)


olist_orders_raw (99441, 8)


olist_order_items_raw (112650, 7)


olist_products_raw (32951, 9)
olist_sellers_raw (3095, 4)


### Typed-таблиці

In [4]:
%%sql
DROP TABLE IF EXISTS olist_order_items CASCADE;
DROP TABLE IF EXISTS olist_orders CASCADE;
DROP TABLE IF EXISTS olist_customers CASCADE;
DROP TABLE IF EXISTS olist_products CASCADE;
DROP TABLE IF EXISTS olist_sellers CASCADE;

CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN ('created', 'approved', 'invoiced', 'processing',
                            'shipped', 'delivered', 'unavailable', 'canceled'))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);

SELECT t.table_name, COUNT(c.column_name) AS n_columns
FROM information_schema.tables AS t
JOIN information_schema.columns AS c ON c.table_name = t.table_name AND c.table_schema = t.table_schema
WHERE t.table_schema = 'public'
  AND t.table_name LIKE 'olist%'
  AND t.table_name NOT LIKE '%raw'
GROUP BY t.table_name
ORDER BY t.table_name;

table_name,n_columns
olist_customers,5
olist_order_items,7
olist_orders,8
olist_products,9
olist_sellers,4


Перенесення raw → typed і кількість рядків у raw-шарі. Колонки `product_name_lenght` і `product_description_lenght` у CSV написані з помилкою, у typed-схемі вони нормалізовані.

In [5]:
%%sql
INSERT INTO olist_customers
SELECT customer_id, customer_unique_id, customer_zip_code_prefix::INTEGER, customer_city, customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT order_id, customer_id, order_status,
       order_purchase_timestamp::TIMESTAMP,
       order_approved_at::TIMESTAMP,
       order_delivered_carrier_date::TIMESTAMP,
       order_delivered_customer_date::TIMESTAMP,
       order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT product_id, product_category_name,
       product_name_lenght::INTEGER,
       product_description_lenght::INTEGER,
       product_photos_qty::INTEGER,
       product_weight_g::INTEGER,
       product_length_cm::INTEGER,
       product_height_cm::INTEGER,
       product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT seller_id, seller_zip_code_prefix::INTEGER, seller_city, seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT order_id, order_item_id::INTEGER, product_id, seller_id,
       shipping_limit_date::TIMESTAMP,
       price::NUMERIC(12, 2),
       freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;

ANALYZE olist_customers, olist_orders, olist_order_items, olist_products, olist_sellers;

SELECT 'customers' AS table_name, COUNT(*) AS n_raw FROM olist_customers_raw
UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders_raw
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items_raw
UNION ALL SELECT 'products', COUNT(*) FROM olist_products_raw
UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers_raw
ORDER BY table_name;

table_name,n_raw
customers,99441
order_items,112650
orders,99441
products,32951
sellers,3095


### Smoke-test кількості рядків

In [6]:
%%sql
SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM olist_customers
UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL SELECT 'products', COUNT(*) FROM olist_products
UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers
ORDER BY table_name;

table_name,n_rows
customers,99441
order_items,112650
orders,99441
products,32951
sellers,3095


**Interpretation:** Кількість рядків у typed-таблицях збігається з raw: 99 441 клієнт і замовлення, 112 650 позицій, 32 951 товар, 3 095 продавців. Втрат немає: усі FK знайшли батьківські записи, а `CHECK` не відхилив жодного значення. Після завантаження виконано `ANALYZE`, щоб планувальник мав статистику для `EXPLAIN` у п. 5.

## 2. CTE-chain pipeline

Кожен stage робить одну трансформацію: Stage 0 — reference date, 1 — сума замовлення, 2 — RFM, 3 — когорта, 4 — rolling AOV за 30 днів, 5 — частка скасувань, 6–7 — улюблена категорія. Фінальний `SELECT` збирає features на рівні `customer_id` через `LEFT JOIN`, тому покупці без замовлень теж залишаються з дефолтними значеннями.

In [7]:
%%sql
WITH reference_date AS (
    -- Stage 0: stable reference date for historical dataset
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + status
    SELECT o.customer_id,
           o.order_id,
           o.order_status,
           o.order_purchase_timestamp AS order_ts,
           o.order_purchase_timestamp::DATE AS order_date,
           COALESCE(SUM(COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    GROUP BY o.customer_id, o.order_id, o.order_status, o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM features per customer
    SELECT s.customer_id,
           (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
           COUNT(*) AS frequency,
           SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
           MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter from first order date
    SELECT r.*,
           DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value per customer
    SELECT s.customer_id,
           s.order_id,
           s.order_ts,
           s.order_total,
           AVG(s.order_total) OVER (
               PARTITION BY s.customer_id
               ORDER BY s.order_ts
               RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
           )::NUMERIC(12, 2) AS avg_order_value_30d,
           ROW_NUMBER() OVER (
               PARTITION BY s.customer_id
               ORDER BY s.order_ts DESC, s.order_id DESC
           ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate per customer
    SELECT customer_id,
           AVG(CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END)::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: revenue by customer and product category
    SELECT o.customer_id,
           COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
           SUM(COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0))::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: preferred category per customer via top-1 ranking
    SELECT customer_id, product_category_name, category_revenue
    FROM (
        SELECT cr.*,
               ROW_NUMBER() OVER (
                   PARTITION BY cr.customer_id
                   ORDER BY cr.category_revenue DESC, cr.product_category_name
               ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT c.customer_id,
       c.customer_state,
       COALESCE(coh.recency_days, 9999) AS recency_days,
       COALESCE(coh.frequency, 0) AS frequency,
       COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
       coh.first_order_date,
       coh.cohort_quarter,
       COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
       COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
       COALESCE(pc.product_category_name, '(none)') AS preferred_category,
       COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov ON aov.customer_id = c.customer_id AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc ON pc.customer_id = c.customer_id
ORDER BY monetary DESC NULLS LAST, c.customer_id
LIMIT 20;

customer_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
1617b1357756262bfa56ab541c47bc16,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0000,telefonia_fixa,13664.08
ec5b2ba62e574342386871631fafd3fc,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0000,telefonia_fixa,7274.88
c6e2731c5b391845f6800c97401a43a9,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0000,utilidades_domesticas,6929.31
f48d464a0baaea338cb25f816991ab1f,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0000,pcs,6922.21
3fd6777bbce08a352fddd04e4a7cc8f6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0000,artes,6726.66
05455dfa7cd02f13d132aa7a6a9729c6,MG,328,1,6081.54,2017-11-24,2017-10-01,6081.54,0.0000,agro_industria_e_comercio,6081.54
df55c14d1476a9a3467f131269c2477f,RJ,565,1,4950.34,2017-04-01,2017-04-01,4950.34,0.0000,eletroportateis,4950.34
e0a2412720e9ea4f26c1ac985f6a7358,GO,98,1,4809.44,2018-07-12,2018-07-01,4809.44,1.0000,cool_stuff,4809.44
24bbf5fd2f2e1b359ee7de94defc4a15,SP,548,1,4764.34,2017-04-18,2017-04-01,4764.34,0.0000,eletroportateis,4764.34
3d979689f636322c62418b6346b1c6d2,PB,118,1,4681.78,2018-06-22,2018-04-01,4681.78,0.0000,pcs,4681.78


**Interpretation:** У топ-20 за `monetary` у всіх `frequency = 1`, а `avg_order_value_30d_recent` і `preferred_category_revenue` дорівнюють `monetary`. Причина в ключі: в Olist `customer_id` створюється для кожного замовлення, тому на цьому рівні в кожного «клієнта» рівно одне замовлення. Також `monetary` враховує скасовані замовлення: `e0a24127…` має 4 809.44 BRL при `cancellation_rate = 1.0000`. Для ML-фіч я б рахував pipeline за `customer_unique_id` і виключав `canceled` з `monetary`.

Перевірка розподілу кількох features по всьому результату pipeline:

In [8]:
%%sql
WITH features AS (
WITH reference_date AS (
    -- Stage 0: stable reference date for historical dataset
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + status
    SELECT o.customer_id,
           o.order_id,
           o.order_status,
           o.order_purchase_timestamp AS order_ts,
           o.order_purchase_timestamp::DATE AS order_date,
           COALESCE(SUM(COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    GROUP BY o.customer_id, o.order_id, o.order_status, o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM features per customer
    SELECT s.customer_id,
           (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
           COUNT(*) AS frequency,
           SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
           MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter from first order date
    SELECT r.*,
           DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value per customer
    SELECT s.customer_id,
           s.order_id,
           s.order_ts,
           s.order_total,
           AVG(s.order_total) OVER (
               PARTITION BY s.customer_id
               ORDER BY s.order_ts
               RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
           )::NUMERIC(12, 2) AS avg_order_value_30d,
           ROW_NUMBER() OVER (
               PARTITION BY s.customer_id
               ORDER BY s.order_ts DESC, s.order_id DESC
           ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate per customer
    SELECT customer_id,
           AVG(CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END)::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: revenue by customer and product category
    SELECT o.customer_id,
           COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
           SUM(COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0))::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: preferred category per customer via top-1 ranking
    SELECT customer_id, product_category_name, category_revenue
    FROM (
        SELECT cr.*,
               ROW_NUMBER() OVER (
                   PARTITION BY cr.customer_id
                   ORDER BY cr.category_revenue DESC, cr.product_category_name
               ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT c.customer_id,
       c.customer_state,
       COALESCE(coh.recency_days, 9999) AS recency_days,
       COALESCE(coh.frequency, 0) AS frequency,
       COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
       coh.first_order_date,
       coh.cohort_quarter,
       COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
       COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
       COALESCE(pc.product_category_name, '(none)') AS preferred_category,
       COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov ON aov.customer_id = c.customer_id AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc ON pc.customer_id = c.customer_id
)
SELECT COUNT(*) AS n_customers,
       COUNT(*) FILTER (WHERE frequency > 1) AS n_frequency_gt_1,
       COUNT(*) FILTER (WHERE cancellation_rate > 0) AS n_with_cancel,
       COUNT(*) FILTER (WHERE preferred_category = '(none)') AS n_no_category,
       MIN(cohort_quarter) AS first_cohort,
       MAX(cohort_quarter) AS last_cohort
FROM features;

n_customers,n_frequency_gt_1,n_with_cancel,n_no_category,first_cohort,last_cohort
99441,0,625,775,2016-07-01,2018-10-01


**Interpretation:** Pipeline повертає 99 441 рядок, по одному на `customer_id`. Жоден не має `frequency > 1`, що підтверджує обмеження ключа. 625 клієнтів мають ненульовий `cancellation_rate` (це всі `canceled`-замовлення), а 775 отримали `preferred_category = '(none)'`, бо їхні замовлення не мають жодної позиції. Когорти охоплюють квартали від 2016-07-01 до 2018-10-01.

## 3. Recursive CTE для referral chain

In [9]:
%%sql
DROP TABLE IF EXISTS hw5_referrals CASCADE;

CREATE TABLE hw5_referrals (
    referrer_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_at          TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    PRIMARY KEY (referred_customer_id),
    CHECK (referrer_customer_id <> referred_customer_id)
);

WITH ordered_customers AS (
    SELECT customer_id, ROW_NUMBER() OVER (ORDER BY customer_id) AS rn
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 10
)
INSERT INTO hw5_referrals (referrer_customer_id, referred_customer_id)
SELECT prev.customer_id AS referrer_customer_id,
       nxt.customer_id AS referred_customer_id
FROM ordered_customers AS prev
JOIN ordered_customers AS nxt ON nxt.rn = prev.rn + 1
ON CONFLICT (referred_customer_id) DO NOTHING;

SELECT referrer_customer_id, referred_customer_id
FROM hw5_referrals
ORDER BY referrer_customer_id, referred_customer_id;

referrer_customer_id,referred_customer_id
00012a2ce6f8dcda20d059ce98491703,000161a058600d5901f007fab4c27140
000161a058600d5901f007fab4c27140,0001fd6190edaaf884bcaf3d49edf079
0001fd6190edaaf884bcaf3d49edf079,0002414f95344307404f0ace7a26f1d5
0002414f95344307404f0ace7a26f1d5,000379cdec625522490c315e70c7a9fb
000379cdec625522490c315e70c7a9fb,0004164d20a9e969af783496f3408652
0004164d20a9e969af783496f3408652,000419c5494106c306a97b5635748086
000419c5494106c306a97b5635748086,00046a560d407e99b969756e0b10f282
00046a560d407e99b969756e0b10f282,00050bf6e01e69d5c0fd612f1bcfb69c
00050bf6e01e69d5c0fd612f1bcfb69c,000598caf2ef4117407665ac33275130


**Interpretation:** Перші 10 клієнтів за `customer_id` утворюють ланцюжок із 9 пар: кожен наступний запрошений попереднім. `PRIMARY KEY (referred_customer_id)` гарантує, що в кожного клієнта не більше одного referrer, а `CHECK` забороняє запросити самого себе. `ON CONFLICT DO NOTHING` робить вставку повторюваною.

In [10]:
%%sql
WITH RECURSIVE referral_depth AS (
    -- ANCHOR: root customers, які не були referred іншими customers
    SELECT c.customer_id,
           1 AS depth,
           ARRAY[c.customer_id]::TEXT[] AS path_ids,
           c.customer_id::TEXT AS path_text
    FROM olist_customers AS c
    WHERE NOT EXISTS (
        SELECT 1 FROM hw5_referrals AS r WHERE r.referred_customer_id = c.customer_id
    )

    UNION ALL

    -- RECURSIVE: перейти від referrer до його referred customers
    SELECT r.referred_customer_id AS customer_id,
           rd.depth + 1 AS depth,
           rd.path_ids || r.referred_customer_id AS path_ids,
           rd.path_text || ' -> ' || r.referred_customer_id AS path_text
    FROM hw5_referrals AS r
    JOIN referral_depth AS rd ON rd.customer_id = r.referrer_customer_id
    WHERE NOT r.referred_customer_id = ANY (rd.path_ids)
      AND rd.depth < 50
)
SELECT depth, COUNT(*) AS n_customers, MAX(path_text) AS sample_path
FROM referral_depth
GROUP BY depth
ORDER BY depth;

depth,n_customers,sample_path
1,99432,ffffe8b65bbe3087b653a978c870db99
2,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140
3,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079
4,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5
5,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb
6,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb -> 0004164d20a9e969af783496f3408652
7,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb -> 0004164d20a9e969af783496f3408652 -> 000419c5494106c306a97b5635748086
8,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb -> 0004164d20a9e969af783496f3408652 -> 000419c5494106c306a97b5635748086 -> 00046a560d407e99b969756e0b10f282
9,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb -> 0004164d20a9e969af783496f3408652 -> 000419c5494106c306a97b5635748086 -> 00046a560d407e99b969756e0b10f282 -> 00050bf6e01e69d5c0fd612f1bcfb69c
10,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb -> 0004164d20a9e969af783496f3408652 -> 000419c5494106c306a97b5635748086 -> 00046a560d407e99b969756e0b10f282 -> 00050bf6e01e69d5c0fd612f1bcfb69c -> 000598caf2ef4117407665ac33275130


**Interpretation:** Anchor через `NOT EXISTS` дає 99 432 корені (усі клієнти, крім 9 запрошених) на `depth = 1`. Далі recursive part проходить ланцюжок по одному клієнту на рівень до `depth = 10`, а `path_text` показує повний шлях від кореня. Рекурсія зупиняється, коли на черговому кроці не знайдено нових `referred_customer_id`; `path_ids` і `depth < 50` додатково захищають від циклу.

## 4. Materialized View з `CONCURRENT REFRESH`

MV містить повний pipeline з п. 2 без `ORDER BY` / `LIMIT`.

In [11]:
%%sql
DROP MATERIALIZED VIEW IF EXISTS mv_olist_customer_features CASCADE;

CREATE MATERIALIZED VIEW mv_olist_customer_features AS
WITH reference_date AS (
    -- Stage 0: stable reference date for historical dataset
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + status
    SELECT o.customer_id,
           o.order_id,
           o.order_status,
           o.order_purchase_timestamp AS order_ts,
           o.order_purchase_timestamp::DATE AS order_date,
           COALESCE(SUM(COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    GROUP BY o.customer_id, o.order_id, o.order_status, o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM features per customer
    SELECT s.customer_id,
           (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
           COUNT(*) AS frequency,
           SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
           MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter from first order date
    SELECT r.*,
           DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value per customer
    SELECT s.customer_id,
           s.order_id,
           s.order_ts,
           s.order_total,
           AVG(s.order_total) OVER (
               PARTITION BY s.customer_id
               ORDER BY s.order_ts
               RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
           )::NUMERIC(12, 2) AS avg_order_value_30d,
           ROW_NUMBER() OVER (
               PARTITION BY s.customer_id
               ORDER BY s.order_ts DESC, s.order_id DESC
           ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate per customer
    SELECT customer_id,
           AVG(CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END)::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: revenue by customer and product category
    SELECT o.customer_id,
           COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
           SUM(COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0))::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: preferred category per customer via top-1 ranking
    SELECT customer_id, product_category_name, category_revenue
    FROM (
        SELECT cr.*,
               ROW_NUMBER() OVER (
                   PARTITION BY cr.customer_id
                   ORDER BY cr.category_revenue DESC, cr.product_category_name
               ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT c.customer_id,
       c.customer_state,
       COALESCE(coh.recency_days, 9999) AS recency_days,
       COALESCE(coh.frequency, 0) AS frequency,
       COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
       coh.first_order_date,
       coh.cohort_quarter,
       COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
       COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
       COALESCE(pc.product_category_name, '(none)') AS preferred_category,
       COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov ON aov.customer_id = c.customer_id AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc ON pc.customer_id = c.customer_id;

CREATE UNIQUE INDEX idx_mv_olist_customer_features_customer_id
    ON mv_olist_customer_features (customer_id);

SELECT COUNT(*) AS n_rows FROM mv_olist_customer_features;

n_rows
99441


In [12]:
%%sql
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST, customer_id
LIMIT 5;

customer_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
1617b1357756262bfa56ab541c47bc16,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0000,telefonia_fixa,13664.08
ec5b2ba62e574342386871631fafd3fc,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0000,telefonia_fixa,7274.88
c6e2731c5b391845f6800c97401a43a9,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0000,utilidades_domesticas,6929.31
f48d464a0baaea338cb25f816991ab1f,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0000,pcs,6922.21
3fd6777bbce08a352fddd04e4a7cc8f6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0000,artes,6726.66


`REFRESH ... CONCURRENTLY` окремою клітинкою: він працює лише за наявності `UNIQUE INDEX` на колонках MV і не блокує читання під час оновлення.

In [13]:
%%sql
REFRESH MATERIALIZED VIEW CONCURRENTLY mv_olist_customer_features;

SELECT COUNT(*) AS n_rows, COUNT(DISTINCT customer_id) AS n_customers
FROM mv_olist_customer_features;

n_rows,n_customers
99441,99441


**Interpretation:** MV містить 99 441 рядок, тобто той самий повний результат pipeline, а її топ-5 збігається з п. 2. `REFRESH MATERIALIZED VIEW CONCURRENTLY` пройшов без помилок, бо `idx_mv_olist_customer_features_customer_id` — повний `UNIQUE INDEX` на колонці MV; після оновлення кількість рядків і унікальних `customer_id` однакова.

## 5. `EXPLAIN (ANALYZE, BUFFERS)`

### 5.1. Повне переобчислення CTE-pipeline (Stage 0–2)

In [14]:
%%sql
EXPLAIN (ANALYZE, BUFFERS)
WITH reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    SELECT o.customer_id,
           o.order_id,
           o.order_status,
           o.order_purchase_timestamp AS order_ts,
           o.order_purchase_timestamp::DATE AS order_date,
           COALESCE(SUM(COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    GROUP BY o.customer_id, o.order_id, o.order_status, o.order_purchase_timestamp
),
stage2_rfm AS (
    SELECT s.customer_id,
           (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
           COUNT(*) AS frequency,
           SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
           MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
)
SELECT *
FROM stage2_rfm
ORDER BY monetary DESC NULLS LAST
LIMIT 10;

QUERY PLAN
Limit (cost=33611.42..33611.45 rows=10 width=65) (actual time=337.247..337.254 rows=10 loops=1)
"Buffers: shared hit=105261, temp read=961 written=1696"
-> Sort (cost=33611.42..33611.92 rows=200 width=65) (actual time=337.246..337.251 rows=10 loops=1)
Sort Key: stage2_rfm.monetary DESC NULLS LAST
Sort Method: top-N heapsort Memory: 27kB
"Buffers: shared hit=105261, temp read=961 written=1696"
-> Subquery Scan on stage2_rfm (cost=33601.60..33607.10 rows=200 width=65) (actual time=253.244..320.533 rows=99441 loops=1)
"Buffers: shared hit=105261, temp read=961 written=1696"
-> HashAggregate (cost=33601.60..33605.10 rows=200 width=69) (actual time=253.243..313.418 rows=99441 loops=1)
"Group Key: o.customer_id, (((max(olist_orders.order_purchase_timestamp))::date + 1))"


### 5.2. Повний pipeline (Stage 0–7) для порівняння

Шаблон 5.1 рахує лише RFM. Щоб побачити вартість window functions і улюбленої категорії, запускаємо той самий pipeline, що лежить в основі MV.

In [15]:
%%sql
EXPLAIN (ANALYZE, BUFFERS)
WITH reference_date AS (
    -- Stage 0: stable reference date for historical dataset
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + status
    SELECT o.customer_id,
           o.order_id,
           o.order_status,
           o.order_purchase_timestamp AS order_ts,
           o.order_purchase_timestamp::DATE AS order_date,
           COALESCE(SUM(COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    GROUP BY o.customer_id, o.order_id, o.order_status, o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM features per customer
    SELECT s.customer_id,
           (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
           COUNT(*) AS frequency,
           SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
           MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter from first order date
    SELECT r.*,
           DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value per customer
    SELECT s.customer_id,
           s.order_id,
           s.order_ts,
           s.order_total,
           AVG(s.order_total) OVER (
               PARTITION BY s.customer_id
               ORDER BY s.order_ts
               RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
           )::NUMERIC(12, 2) AS avg_order_value_30d,
           ROW_NUMBER() OVER (
               PARTITION BY s.customer_id
               ORDER BY s.order_ts DESC, s.order_id DESC
           ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate per customer
    SELECT customer_id,
           AVG(CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END)::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: revenue by customer and product category
    SELECT o.customer_id,
           COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
           SUM(COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0))::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: preferred category per customer via top-1 ranking
    SELECT customer_id, product_category_name, category_revenue
    FROM (
        SELECT cr.*,
               ROW_NUMBER() OVER (
                   PARTITION BY cr.customer_id
                   ORDER BY cr.category_revenue DESC, cr.product_category_name
               ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT c.customer_id,
       c.customer_state,
       COALESCE(coh.recency_days, 9999) AS recency_days,
       COALESCE(coh.frequency, 0) AS frequency,
       COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
       coh.first_order_date,
       coh.cohort_quarter,
       COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
       COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
       COALESCE(pc.product_category_name, '(none)') AS preferred_category,
       COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov ON aov.customer_id = c.customer_id AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc ON pc.customer_id = c.customer_id
ORDER BY monetary DESC NULLS LAST, c.customer_id
LIMIT 20;

QUERY PLAN
Limit (cost=116352.33..116352.38 rows=20 width=150) (actual time=1322.656..1323.979 rows=20 loops=1)
"Buffers: shared hit=111711, temp read=8395 written=8544"
CTE stage1_orders_total
-> GroupAggregate (cost=0.83..27070.14 rows=99441 width=103) (actual time=0.035..178.297 rows=99441 loops=1)
Group Key: o_1.order_id
Buffers: shared hit=103453
-> Merge Left Join (cost=0.83..24485.04 rows=112650 width=95) (actual time=0.021..91.958 rows=113425 loops=1)
Merge Cond: (o_1.order_id = oi_1.order_id)
Buffers: shared hit=103453
-> Index Scan using olist_orders_pkey on olist_orders o_1 (cost=0.42..12564.02 rows=99441 width=83) (actual time=0.009..40.026 rows=99441 loops=1)


### 5.3. Lookup із MV за `customer_id`

In [16]:
%%sql
EXPLAIN (ANALYZE, BUFFERS)
WITH target_customer AS (
    SELECT customer_id
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 1
)
SELECT mv.*
FROM mv_olist_customer_features AS mv
JOIN target_customer AS t ON t.customer_id = mv.customer_id;

QUERY PLAN
Nested Loop (cost=0.83..8.96 rows=1 width=158) (actual time=0.030..0.031 rows=1 loops=1)
Buffers: shared hit=8
-> Limit (cost=0.42..0.53 rows=1 width=33) (actual time=0.017..0.017 rows=1 loops=1)
Buffers: shared hit=4
-> Index Only Scan using olist_customers_pkey on olist_customers (cost=0.42..11128.02 rows=99441 width=33) (actual time=0.016..0.016 rows=1 loops=1)
Heap Fetches: 1
Buffers: shared hit=4
-> Index Scan using idx_mv_olist_customer_features_customer_id on mv_olist_customer_features mv (cost=0.42..8.44 rows=1 width=158) (actual time=0.011..0.012 rows=1 loops=1)
Index Cond: (customer_id = olist_customers.customer_id)
Buffers: shared hit=4


### 5.4. Top-N із MV з індексом на `monetary`

In [17]:
%%sql
CREATE INDEX IF NOT EXISTS idx_mv_olist_customer_features_monetary
    ON mv_olist_customer_features (monetary DESC);

EXPLAIN (ANALYZE, BUFFERS)
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST
LIMIT 10;

QUERY PLAN
Limit (cost=4449.01..4450.16 rows=10 width=158) (actual time=9.411..10.783 rows=10 loops=1)
Buffers: shared hit=1636
-> Gather Merge (cost=4449.01..11175.94 rows=58495 width=158) (actual time=9.409..10.780 rows=10 loops=1)
Workers Planned: 1
Workers Launched: 1
Buffers: shared hit=1636
-> Sort (cost=3449.00..3595.24 rows=58495 width=158) (actual time=8.076..8.077 rows=8 loops=2)
Sort Key: monetary DESC NULLS LAST
Sort Method: top-N heapsort Memory: 26kB
Buffers: shared hit=1636


Індекс `monetary DESC` за замовчуванням упорядкований як `DESC NULLS FIRST`, а запит сортує `DESC NULLS LAST`, тому планувальник не може його використати. Створюємо індекс з тим самим порядком, що й у запиті:

In [18]:
%%sql
DROP INDEX IF EXISTS idx_mv_olist_customer_features_monetary;
CREATE INDEX idx_mv_olist_customer_features_monetary
    ON mv_olist_customer_features (monetary DESC NULLS LAST);

EXPLAIN (ANALYZE, BUFFERS)
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST
LIMIT 10;

QUERY PLAN
Limit (cost=0.42..1.32 rows=10 width=158) (actual time=0.015..0.025 rows=10 loops=1)
Buffers: shared hit=10 read=3
-> Index Scan using idx_mv_olist_customer_features_monetary on mv_olist_customer_features (cost=0.42..8996.03 rows=99441 width=158) (actual time=0.014..0.023 rows=10 loops=1)
Buffers: shared hit=10 read=3
Planning:
Buffers: shared hit=16 read=1
Planning Time: 0.118 ms
Execution Time: 0.034 ms


**Interpretation:** - **5.1, Stage 0–2:** 338 ms. Основна вартість — `GroupAggregate` по `order_id` після `Merge Left Join` двох `Index Scan` і `HashAggregate` по `customer_id`, який не вмістився в `work_mem` (`Batches: 21`, `Disk Usage: 7400kB`).
- **5.2, повний pipeline:** 1329 ms. До плану додаються `CTE Scan on stage1_orders_total` (stage 1 матеріалізується один раз, бо на нього посилаються три stages), два `WindowAgg` з `Sort Method: external merge Disk: 9448kB` для rolling AOV, окремий `WindowAgg` з `Incremental Sort` для улюбленої категорії, ще один `HashAggregate` зі спілом на диск для `cancellation_rate` і ланцюжок `Hash Left Join`.
- **5.3, lookup із MV:** 0.045 ms. Два `Index Scan`, по 4 буфери на кожен.
- **5.4, top-N з MV:** з індексом `monetary DESC` план не змінився — `Parallel Seq Scan` + `top-N heapsort`, 10.8 ms. З індексом `DESC NULLS LAST` — `Index Scan using idx_mv_olist_customer_features_monetary`, 0.034 ms.

## Reflection

**1. Найдорожча feature.** За планом 5.2 це `avg_order_value_30d`: гілка `aov` містить два `WindowAgg` над усіма 99 441 замовленнями і `Sort` з `external merge Disk: 9448kB`, бо сортування за `customer_id, order_ts` не вміщується в `work_mem`. Друга за вартістю — `preferred_category`: `Hash Join` з `order_items` і `products`, `GroupAggregate` з external sort і ще один `WindowAgg`. `HashAggregate` для RFM і `cancellation_rate` теж спілять на диск (`Batches: 21`), але дешевші.

**2. Кешування.** Так. Повний pipeline виконується 1329 ms, а lookup із MV — 0.045 ms: замість `Seq Scan` базових таблиць і агрегацій у плані лише два `Index Scan`, по 4 буфери. Olist історичний, тому вистачить on-demand refresh після завантаження нової партії; у живому магазині — щодоби вночі через `REFRESH ... CONCURRENTLY`, щоб не блокувати читання.

**3. Коли VIEW кращий.** Коли потрібні завжди свіжі дані й запит дешевий: наприклад, статус замовлень за останню годину для операційного дашборду, або фільтр по одному клієнту, який індекс у базових таблицях і так швидко віддає.

**4. Termination recursive CTE.** Рекурсія зупиняється, коли recursive part не повертає нових рядків. При циклі `A → B → A` без захисту CTE ходила б по колу нескінченно. Тут спрацює `NOT r.referred_customer_id = ANY(rd.path_ids)`: `A` вже є в шляху, тому крок `B → A` відкидається. `rd.depth < 50` — запасний обмежувач для довгих або пошкоджених графів. У PostgreSQL 14+ те саме дає `CYCLE customer_id SET is_cycle USING path`.

**5. Trade-off.** Для цього pipeline головні recompute-cost і latency: перерахунок займає понад секунду і спілить на диск, тоді як читання з MV триває частки мілісекунди. Freshness менш критична, бо customer-features для моделі не змінюються щохвилини. Storage-cost малий — близько 100 тис. рядків. Окремий урок із 5.4: з індексом `monetary DESC` план залишився `Parallel Seq Scan` + `top-N heapsort`, і лише індекс `DESC NULLS LAST`, який збігається з `ORDER BY`, дав `Index Scan`.